In [ ]:
import os
import glob
import warnings
import shutil

warnings.filterwarnings("ignore")
shutil.rmtree = lambda *args, **kwargs: None 

import numpy as np
import torch
from nilearn import datasets

print("=== [Cell 1] 뇌 지도(AAL) 강제 연결 모드 ===")

abide = datasets.fetch_abide_pcp(data_dir='./abide_data', pipeline='cpac', n_subjects=None)

search_pattern = './nilearn_data/**/*.nii'
found_nii = glob.glob(search_pattern, recursive=True)

if not found_nii:
    print(" 에러: .nii 파일을 찾을 수 없습니다.")
else:
    selected_aal = found_nii[0]
    print(f" 뇌 지도 파일을 찾았습니다: {selected_aal}")

    class MockAtlas:
        def __init__(self, maps):
            self.maps = maps
    
    atlas = MockAtlas(maps=selected_aal)

=== [Cell 1] 뇌 지도(AAL) 강제 연결 모드 ===
[fetch_abide_pcp] Dataset found in abide_data\ABIDE_pcp
[fetch_abide_pcp] Downloading data from https://s3.amazonaws.com/fcp-indi/data/Projects/ABIDE_Initiative/Outputs/cpac/nofilt_noglobal/func_preproc/Olin_0050115_func_preproc.nii.gz ...
Downloaded 19595264 of 119903539 bytes (16.3%%,  2.0min remaining)
Downloaded 22478848 of 119903539 bytes (18.7%%,   58.1s remaining)
Downloaded 25935872 of 119903539 bytes (21.6%%,   42.2s remaining)
Downloaded 28262400 of 119903539 bytes (23.6%%,   40.8s remaining)
Downloaded 30539776 of 119903539 bytes (25.5%%,   39.8s remaining)
Downloaded 32980992 of 119903539 bytes (27.5%%,   38.3s remaining)
Downloaded 35454976 of 119903539 bytes (29.6%%,   36.8s remaining)
Downloaded 37896192 of 119903539 bytes (31.6%%,   35.5s remaining)
Downloaded 40402944 of 119903539 bytes (33.7%%,   34.1s remaining)
Downloaded 42942464 of 119903539 bytes (35.8%%,   32.7s remaining)
Downloaded 45416448 of 119903539 bytes (37.9%%,   31.5

In [ ]:
import nibabel as nib
import numpy as np

atlas_img = nib.load(atlas.maps)
atlas_data = atlas_img.get_fdata()

num_regions = len(np.unique(atlas_data)) - 1
print(f"Atlas의 실제 구역 개수: {num_regions}개")

In [ ]:
import os
import torch
import numpy as np
import warnings
from torch_geometric.data import Data
from sklearn.preprocessing import StandardScaler
from nilearn.maskers import NiftiLabelsMasker
from nilearn.connectome import ConnectivityMeasure

warnings.filterwarnings("ignore")

save_dir = './preprocessed_data_final'
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

print(f"전처리 시작 (대상: {len(abide.func_preproc)}명)")

masker = NiftiLabelsMasker(labels_img=atlas.maps, standardize=True)
corr_measure = ConnectivityMeasure(kind='correlation')

def preprocess_and_save():
    scaler = StandardScaler()
    
    for i in range(len(abide.func_preproc)):
        try:
            fmri_file = abide.func_preproc[i]
            time_series = masker.fit_transform(fmri_file) 

            corr_matrix = corr_measure.fit_transform([time_series])[0]
            z_matrix = np.arctanh(corr_matrix)
            z_matrix[np.isinf(z_matrix)] = 0  

            mean_val = np.mean(time_series, axis=0).reshape(-1, 1)
            std_val = np.std(time_series, axis=0).reshape(-1, 1)
            connectivity_strength = np.sum(np.abs(corr_matrix), axis=0).reshape(-1, 1)
            
            raw_features = np.hstack([mean_val, std_val, connectivity_strength])
            node_x = torch.tensor(scaler.fit_transform(raw_features), dtype=torch.float)

            edge_indices = np.where(np.abs(corr_matrix) > 0.1)
            edge_index = torch.tensor(np.stack(edge_indices), dtype=torch.long)
            edge_attr = torch.tensor(z_matrix[edge_indices], dtype=torch.float)

            actual_label = abide.phenotypic['DX_GROUP'].iloc[i]
            y = torch.tensor([0 if actual_label == 2 else 1], dtype=torch.long)

            sub_id = abide.phenotypic['SUB_ID'].iloc[i]
            data = Data(x=node_x, edge_index=edge_index, edge_attr=edge_attr, y=y)
            data.sub_id = sub_id

            torch.save(data, os.path.join(save_dir, f'sub_{sub_id}.pt'))
            
            if (i + 1) % 10 == 0:
                print(f" [{i+1}/{len(abide.func_preproc)}] 완료 - ID: {sub_id}")
                
        except Exception as e:
            print(f" {i}번 환자({abide.phenotypic['SUB_ID'].iloc[i]}) 처리 실패: {e}")

preprocess_and_save()
print(f"\n 모든 데이터가 '{save_dir}'에 완벽하게 저장되었습니다.")

In [ ]:
import os
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import random

save_dir = './preprocessed_data_final'
pt_files = [f for f in os.listdir(save_dir) if f.endswith('.pt')]

if pt_files:
    sample_file = random.choice(pt_files)
    data = torch.load(os.path.join(save_dir, sample_file), weights_only=False)

    num_nodes = data.x.shape[0]
    
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))

    sns.kdeplot(data.x[:, 0].numpy(), ax=axes[0], label='Mean', fill=True)
    sns.kdeplot(data.x[:, 1].numpy(), ax=axes[0], label='Std', fill=True)
    sns.kdeplot(data.x[:, 2].numpy(), ax=axes[0], label='Centrality', fill=True)
    axes[0].set_title(f'Node Features (N={num_nodes})')
    axes[0].legend()

    adj = np.zeros((num_nodes, num_nodes))
    row, col = data.edge_index.numpy()
    weights = data.edge_attr.numpy()

    mask = (row < num_nodes) & (col < num_nodes)
    adj[row[mask], col[mask]] = weights[mask]
    
    sns.heatmap(adj, cmap='RdBu_r', center=0, ax=axes[1])
    axes[1].set_title(f'Subject {data.sub_id} Connectivity Matrix')

    sns.histplot(data.edge_attr.numpy(), bins=50, kde=True, color='green', ax=axes[2])
    axes[2].set_title('Edge Weights (Fisher Z) Distribution')
    
    plt.tight_layout()
    plt.show()

    print(f"시각화 완료 (노드 개수: {num_nodes}개 확인)")

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import os
import torch
import numpy as np
import random

print("데이터성능 테스트 시작 (Random Forest Baseline)...")

save_dir = './preprocessed_data_final'
pt_files = [f for f in os.listdir(save_dir) if f.endswith('.pt')]

X_list = []
y_list = []

sample_size = min(len(pt_files), 200)
for fname in random.sample(pt_files, sample_size):
    d = torch.load(os.path.join(save_dir, fname), weights_only=False)

    X_list.append(d.edge_attr.mean().item()) 
    y_list.append(d.y.item())

X = np.array(X_list).reshape(-1, 1)
y = np.array(y_list)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
clf = RandomForestClassifier(random_state=42)
clf.fit(X_train, y_train)
pred = clf.predict(X_test)

acc = accuracy_score(y_test, pred)
print(f"- 단순 통계량만으로 예측한 정확도: {acc*100:.2f}%")

In [ ]:
import os
import matplotlib.pyplot as plt
from nilearn import plotting, image

print("4D fMRI 원본 영상 시각화 준비 중...")

found_files = []
search_path = './abide_data' if os.path.exists('./abide_data') else '.'

for root, dirs, files in os.walk(search_path):
    for file in files:
        if file.endswith('.nii.gz'):
            found_files.append(os.path.join(root, file))

if found_files:
    sample_file = found_files[0]
    print(f"4D fMRI 원본을 찾았습니다: {sample_file}")

    first_frame_img = image.index_img(sample_file, 0)
    
    # 잘라낸 3D 프레임으로 뇌 단면 시각화
    display = plotting.plot_epi(first_frame_img, title="Original 4D fMRI Scan (Frame t=0)")
    plt.show()
    

In [ ]:
import os
import glob
import torch
import numpy as np
import networkx as nx
import pandas as pd 
from nilearn import datasets
from nilearn.maskers import NiftiLabelsMasker
from torch_geometric.data import Data

print(" Schaefer 2018 (400 Nodes) 뇌 지도를 다운로드합니다...")
schaefer = datasets.fetch_atlas_schaefer_2018(n_rois=400, yeo_networks=7, resolution_mm=2)
masker = NiftiLabelsMasker(labels_img=schaefer.maps, standardize=True, memory='nilearn_cache')

save_dir = './dataset_v4_schaefer400'
os.makedirs(save_dir, exist_ok=True)
print("뇌 지도 및 저장 폴더 세팅 완료!\n")


In [ ]:
import os
import glob
import warnings
import shutil

warnings.filterwarnings("ignore")
shutil.rmtree = lambda *args, **kwargs: None 

import numpy as np
import torch
from nilearn import datasets

print("=== [Cell 1] ABIDE 로드 및 Schaefer 400 세팅 ===")

abide = datasets.fetch_abide_pcp(data_dir='./abide_data', pipeline='cpac', n_subjects=None)
print(f" ABIDE 데이터 로드 완료! (총 {len(abide.func_preproc)}명)")

print(" Schaefer 2018 (400 Nodes) 뇌 지도를 세팅합니다...")
schaefer = datasets.fetch_atlas_schaefer_2018(n_rois=400, yeo_networks=7)

class MockAtlas:
    def __init__(self, maps):
        self.maps = maps
        
atlas = MockAtlas(maps=schaefer.maps)

In [ ]:
import os
import torch
import numpy as np
import warnings
from torch_geometric.data import Data
from sklearn.preprocessing import StandardScaler
from nilearn.maskers import NiftiLabelsMasker
from nilearn.connectome import ConnectivityMeasure

warnings.filterwarnings("ignore")

save_dir = './preprocessed_data_v6_schaefer_hard'
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

print(f"전처리 시작 (대상: {len(abide.func_preproc)}명)")

masker = NiftiLabelsMasker(labels_img=atlas.maps, standardize=True, memory='nilearn_cache')
corr_measure = ConnectivityMeasure(kind='correlation')

def preprocess_and_save():
    scaler = StandardScaler()
    
    for i in range(len(abide.func_preproc)):
        try:
            fmri_file = abide.func_preproc[i]
            time_series = masker.fit_transform(fmri_file) 

            corr_matrix = corr_measure.fit_transform([time_series])[0]
            corr_matrix_clipped = np.clip(corr_matrix, -0.999, 0.999) 
            z_matrix = np.arctanh(corr_matrix_clipped)
            z_matrix[np.isinf(z_matrix)] = 0  

            mean_val = np.mean(time_series, axis=0).reshape(-1, 1)
            std_val = np.std(time_series, axis=0).reshape(-1, 1)
            connectivity_strength = np.sum(np.abs(corr_matrix), axis=0).reshape(-1, 1)
            
            raw_features = np.hstack([mean_val, std_val, connectivity_strength])
            node_x = torch.tensor(scaler.fit_transform(raw_features), dtype=torch.float)

            edge_indices = np.where(np.abs(corr_matrix) > 0.35)
            edge_index = torch.tensor(np.stack(edge_indices), dtype=torch.long)
            edge_attr = torch.tensor(z_matrix[edge_indices], dtype=torch.float).view(-1, 1)

            actual_label = abide.phenotypic['DX_GROUP'].iloc[i]
            y = torch.tensor([0 if actual_label == 2 else 1], dtype=torch.long)
            
            sub_id = abide.phenotypic['SUB_ID'].iloc[i]
            data = Data(x=node_x, edge_index=edge_index, edge_attr=edge_attr, y=y)
            data.sub_id = sub_id

            torch.save(data, os.path.join(save_dir, f'sub_{sub_id}.pt'))
            
            if (i + 1) % 10 == 0:
                print(f" [{i+1}/{len(abide.func_preproc)}] 완료 - ID: {sub_id} (남은 엣지: {edge_index.shape[1]}개)")
                
        except Exception as e:
            print(f" {i}번 환자 처리 실패: {e}")

preprocess_and_save()
print(f"\n최종 데이터가 '{save_dir}'에 저장되었습니다.")

In [ ]:
import os
import glob
import torch
import numpy as np
from torch_geometric.data import Data 
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

data_dir = r'C:\Users\rty76\OneDrive\바탕 화면\ASD_project\전처리 2'
pt_files = glob.glob(os.path.join(data_dir, '*.pt'))

print(f" '{data_dir}' 폴더에서 총 {len(pt_files)}개의 .pt 파일을 찾았습니다!")

if len(pt_files) == 0:
    print(" 파일을 못 찾았습니다. 경로를 다시 확인해주세요.")
else:
    X_list = []
    y_list = []

    for file in pt_files:
        try:
            data = torch.load(file, weights_only=False) 

            flat_features = data.x.numpy().flatten() 
            
            X_list.append(flat_features)
            y_list.append(data.y.item())
        except Exception as e:
            print(f" 파일 읽기 에러 ({os.path.basename(file)}): {e}")

    X = np.array(X_list)
    y = np.array(y_list)

    if len(X) == 0:
        print("\n 데이터 변환에 실패했습니다.")
    else:
        print(f" 1차원 변환 완료. 환자 수: {X.shape[0]}명 / 1명당 특징 수: {X.shape[1]}개")

        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

        print(" 랜덤 포레스트가 뇌 특징을 학습하고 있습니다... (약 5~10초 소요)")
        rf_model = RandomForestClassifier(n_estimators=500, random_state=42, n_jobs=-1, class_weight='balanced')
        rf_model.fit(X_train, y_train)

        y_pred = rf_model.predict(X_test)
        accuracy = accuracy_score(y_test, y_pred)

        print("\n========================================")
        print(f" 랜덤 포레스트 베이스라인 정확도: {accuracy * 100:.2f}%")
        print("========================================")
        print("\n 상세 성적표:")
        print(classification_report(y_test, y_pred, target_names=['TC (정상)', 'ASD (자폐)']))

In [ ]:
import os
import glob
import torch
import torch.nn.functional as F
from torch.nn import Linear
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool
from sklearn.model_selection import train_test_split

data_dir = r'C:\Users\rty76\OneDrive\바탕 화면\ASD_project\전처리 2'
pt_files = glob.glob(os.path.join(data_dir, '*.pt'))

print(f" 데이터를 불러옵니다... (총 {len(pt_files)}명)")

dataset = []
for file in pt_files:
    try:
        data = torch.load(file, weights_only=False)
        dataset.append(data)
    except Exception as e:
        pass

train_dataset, test_dataset = train_test_split(dataset, test_size=0.2, random_state=42)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

class LightGCN(torch.nn.Module):
    def __init__(self):
        super(LightGCN, self).__init__()
        self.conv1 = GCNConv(3, 16)
        self.conv2 = GCNConv(16, 16)
        self.lin = Linear(16, 2)

    def forward(self, x, edge_index, batch):
        x = self.conv1(x, edge_index)
        x = F.relu(x) 
        x = self.conv2(x, edge_index)

        x = global_mean_pool(x, batch)

        x = F.dropout(x, p=0.5, training=self.training)
        x = self.lin(x)
        return x

model = LightGCN()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01) 
criterion = torch.nn.CrossEntropyLoss() 

def train():
    model.train()
    total_loss = 0
    for data in train_loader:
        optimizer.zero_grad() 
        out = model(data.x, data.edge_index, data.batch) 
        loss = criterion(out, data.y) 
        loss.backward() 
        optimizer.step() 
        total_loss += loss.item() * data.num_graphs
    return total_loss / len(train_dataset)

def test(loader):
    model.eval()
    correct = 0
    for data in loader:
        out = model(data.x, data.edge_index, data.batch)
        pred = out.argmax(dim=1) # 확률 제일 높은 거 고르기
        correct += int((pred == data.y).sum())
    return correct / len(loader.dataset)

print("\n [GCN 엔진 가동] 학습을 시작합니다!")
for epoch in range(1, 51):
    loss = train()
    if epoch % 10 == 0:
        train_acc = test(train_loader)
        test_acc = test(test_loader)
        print(f"🔄 Epoch {epoch:03d} | 오답률(Loss): {loss:.4f} | 학습 정확도: {train_acc*100:.1f}% | 테스트 정확도: {test_acc*100:.1f}%")

print("\n========================================")
print(f" 최종 GCN 테스트 정확도: {test(test_loader)*100:.2f}%")
print("========================================")

In [ ]:
import os
import numpy as np
import torch
import torch.nn.functional as F
from torch.nn import Linear, Sequential, ReLU
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, GINConv, global_mean_pool
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
import warnings
warnings.filterwarnings('ignore')

class BrainGCN(torch.nn.Module):
    
    def __init__(self, in_dim=3, hidden_dim=64, num_classes=2): 
        super(BrainGCN, self).__init__()
        self.conv1 = GCNConv(in_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.fc = Linear(hidden_dim, num_classes)
    def forward(self, x, edge_index, edge_attr, batch):
        
        edge_weight = edge_attr.squeeze() if edge_attr is not None else None
        
        x = self.conv1(x, edge_index, edge_weight=edge_weight)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)
        
        x = self.conv2(x, edge_index, edge_weight=edge_weight)
        x = F.relu(x)
        
        x = global_mean_pool(x, batch)
        return self.fc(x)

class BrainGIN(torch.nn.Module):
    
    def __init__(self, in_dim=3, hidden_dim=64, num_classes=2):
        super(BrainGIN, self).__init__()
        nn1 = Sequential(Linear(in_dim, hidden_dim), ReLU(), Linear(hidden_dim, hidden_dim))
        self.conv1 = GINConv(nn1)
        
        nn2 = Sequential(Linear(hidden_dim, hidden_dim), ReLU(), Linear(hidden_dim, hidden_dim))
        self.conv2 = GINConv(nn2)
        
        self.fc = Linear(hidden_dim, num_classes)
    def forward(self, x, edge_index, edge_attr, batch):
        
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)
        
        x = self.conv2(x, edge_index)
        x = F.relu(x)
        
        x = global_mean_pool(x, batch)
        return self.fc(x)

def load_v8_data(data_dir):
    print(f" [{data_dir}] 경로에서 V8 전체 데이터 로드 중...")
    data_list = []
    
    for file_name in os.listdir(data_dir):
        if file_name.startswith('sub_'): 
            try:
                
                data = torch.load(os.path.join(data_dir, file_name), map_location='cpu', weights_only=False)
                data_list.append(data)
            except Exception as e:
                print(f" {file_name} 로드 실패: {e}")
                continue
                
    print(f" 총 {len(data_list)}명 전체 데이터 로드 완료!\n")
    return data_list

def evaluate_model(data_list, model_type="GCN", epochs=50):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    labels = [int(data.y.item()) for data in data_list]
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    acc_scores, f1_scores = [], []
    print(f" [{model_type}] 모델 5-Fold 교차 검증 시작 ")

    for fold, (train_idx, test_idx) in enumerate(skf.split(data_list, labels)):
        train_data = [data_list[i] for i in train_idx]
        test_data = [data_list[i] for i in test_idx]
        
        train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
        test_loader = DataLoader(test_data, batch_size=32, shuffle=False)
        
        input_dim = data_list[0].num_features 
        
        if model_type == "GCN":
            model = BrainGCN(in_dim=input_dim, hidden_dim=64, num_classes=2).to(device)
        elif model_type == "GIN":
            model = BrainGIN(in_dim=input_dim, hidden_dim=64, num_classes=2).to(device)
            
        optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
        criterion = torch.nn.CrossEntropyLoss()

        model.train()
        for epoch in range(epochs):
            for batch in train_loader:
                batch = batch.to(device)
                optimizer.zero_grad()
                out = model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)
                loss = criterion(out, batch.y)
                loss.backward()
                optimizer.step()

        model.eval()
        preds, trues = [], []
        with torch.no_grad():
            for batch in test_loader:
                batch = batch.to(device)
                out = model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)
                preds.extend(out.argmax(dim=1).cpu().numpy())
                trues.extend(batch.y.cpu().numpy())
                
        acc = accuracy_score(trues, preds)
        f1 = f1_score(trues, preds, average='macro')
        acc_scores.append(acc)
        f1_scores.append(f1)
        print(f"Fold {fold+1} | Accuracy: {acc:.4f} | F1 Score: {f1:.4f}")
        
    print(f" [{model_type}] 최종 평균 -> Accuracy: {np.mean(acc_scores):.4f} | F1: {np.mean(f1_scores):.4f}\n")
    return np.mean(acc_scores), np.mean(f1_scores)

if __name__ == "__main__":
    V8_DATA_DIR = r'C:\Users\rty76\OneDrive\바탕 화면\ASD_project\전처리 2'

    all_data = load_v8_data(V8_DATA_DIR)

    gcn_acc, gcn_f1 = evaluate_model(all_data, model_type="GCN", epochs=50)
    
    gin_acc, gin_f1 = evaluate_model(all_data, model_type="GIN", epochs=50)
    
    print("====================================================")
    print(f"- GCN | Accuracy: {gcn_acc:.4f} | F1: {gcn_f1:.4f}")
    print(f"- GIN | Accuracy: {gin_acc:.4f} | F1: {gin_f1:.4f}")
    print("====================================================")

In [ ]:
import os
import shutil
import pandas as pd

BASE_DIR = r'C:\Users\rty76\OneDrive\바탕 화면\ASD_project'
PHENO_CSV = os.path.join(BASE_DIR, r'C:\Users\rty76\OneDrive\바탕 화면\ASD_project\abide_data\ABIDE_pcp\Phenotypic_V1_0b_preprocessed1.csv')

V8_SCHAEFER_DIR = os.path.join(BASE_DIR, '전처리 2')          
AAL_OLD_DIR = os.path.join(BASE_DIR, 'preprocessed_data_final') 

DIR_SC_CHILD = os.path.join(BASE_DIR, 'data_Schaefer_Adolescent')
DIR_SC_ADULT = os.path.join(BASE_DIR, 'data_Schaefer_Adult')
DIR_AAL_CHILD = os.path.join(BASE_DIR, 'data_AAL_Adolescent')
DIR_AAL_ADULT = os.path.join(BASE_DIR, 'data_AAL_Adult')

for d in [DIR_SC_CHILD, DIR_SC_ADULT, DIR_AAL_CHILD, DIR_AAL_ADULT]:
    os.makedirs(d, exist_ok=True)

df = pd.read_csv(PHENO_CSV)
age_dict = dict(zip(df['SUB_ID'], df['AGE_AT_SCAN']))

def split_data_by_age(src_dir, child_dir, adult_dir, desc):
    print(f" [{desc}] 데이터 연령별 분할을 시작합니다...")
    if not os.path.exists(src_dir):
        print(f" 원본 폴더를 찾을 수 없습니다: {src_dir}")
        return
    
    child_cnt, adult_cnt = 0, 0
    
    for file_name in os.listdir(src_dir):
        if file_name.startswith('sub_'):
            try:
                sub_id = int(file_name.replace('sub_', '').replace('.pt', ''))
                age = age_dict.get(sub_id, None)
                if age is None: continue
                
                src_path = os.path.join(src_dir, file_name)
                if age <= 18.0:
                    shutil.copy(src_path, os.path.join(child_dir, file_name))
                    child_cnt += 1
                else:
                    shutil.copy(src_path, os.path.join(adult_dir, file_name))
                    adult_cnt += 1
            except Exception as e:
                continue
                
    print(f" {desc} 분류 완료. 청소년: {child_cnt}명 | 성인: {adult_cnt}명\n")

if __name__ == "__main__":
    split_data_by_age(V8_SCHAEFER_DIR, DIR_SC_CHILD, DIR_SC_ADULT, "Schaefer 400")
    split_data_by_age(AAL_OLD_DIR, DIR_AAL_CHILD, DIR_AAL_ADULT, "AAL 116")


In [ ]:
import os
import torch
import warnings
warnings.filterwarnings('ignore')

BASE_DIR = r'C:\Users\rty76\OneDrive\바탕 화면\ASD_project'
K_VAL = 20 

src_dirs = [
    'data_Schaefer_Adolescent', 'data_Schaefer_Adult',
    'data_AAL_Adolescent', 'data_AAL_Adult'
]

def process_and_save_topk(base_dir, folder_name, k):
    src_path = os.path.join(base_dir, folder_name)
    dst_path = os.path.join(base_dir, f"{folder_name}_Top{k}") 
    
    if not os.path.exists(src_path): return
    os.makedirs(dst_path, exist_ok=True)
    
    files = [f for f in os.listdir(src_path) if f.startswith('sub_')]
    print(f" [{folder_name}] -> Top-{k} 변환 및 [{dst_path}]에 저장 중...")
    
    success_cnt = 0
    for file_name in files:
        try:
            data = torch.load(os.path.join(src_path, file_name), map_location='cpu', weights_only=False)
            num_nodes = data.x.size(0)

            adj = torch.zeros((num_nodes, num_nodes), dtype=torch.float32)
            adj[data.edge_index[0], data.edge_index[1]] = data.edge_attr.squeeze()
            
            new_edge_index = []
            new_edge_attr = []

            for i in range(num_nodes):
                vals, indices = torch.topk(torch.abs(adj[i]), k)
                for val, idx in zip(vals, indices):
                    original_val = adj[i, idx]
                    if original_val != 0:
                        new_edge_index.append([i, idx.item()])
                        new_edge_attr.append(original_val.item())

            data.edge_index = torch.tensor(new_edge_index, dtype=torch.long).t().contiguous()
            data.edge_attr = torch.tensor(new_edge_attr, dtype=torch.float32).unsqueeze(1)

            torch.save(data, os.path.join(dst_path, file_name))
            success_cnt += 1
            
        except Exception as e:
            continue
            
    print(f" {success_cnt}명 저장 완료. \n")

if __name__ == "__main__":
    for folder in src_dirs:
        process_and_save_topk(BASE_DIR, folder, K_VAL)
        
    print(f" [모든 데이터가 Top-{K_VAL}로 영구 변환되어 새 폴더에 저장되었습니다.]")

In [ ]:
import os
import numpy as np
import torch
import torch.nn.functional as F
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATConv, global_mean_pool
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
import warnings
warnings.filterwarnings('ignore')

class BrainGAT(torch.nn.Module):
    def __init__(self, in_dim, hidden_dim=64, num_classes=2, heads=4):
        super(BrainGAT, self).__init__()
        self.conv1 = GATConv(in_dim, hidden_dim, heads=heads, dropout=0.6)
        self.conv2 = GATConv(hidden_dim * heads, hidden_dim, heads=1, concat=False, dropout=0.6)
        self.fc = torch.nn.Linear(hidden_dim, num_classes)

    def forward(self, x, edge_index, edge_attr, batch):
        x = F.dropout(x, p=0.6, training=self.training)
        x = F.elu(self.conv1(x, edge_index)) 
        x = F.dropout(x, p=0.6, training=self.training)
        x = F.elu(self.conv2(x, edge_index))

        x = global_mean_pool(x, batch)
        return self.fc(x)

def load_prepared_data(data_dir):
    print(f" [{data_dir}] 경로에서 데이터를 로드합니다...")
    data_list = []
    if not os.path.exists(data_dir):
        print(f" 폴더를 찾을 수 없습니다: {data_dir}")
        return data_list
        
    for file_name in os.listdir(data_dir):
        if file_name.startswith('sub_'):
            try:
                data = torch.load(os.path.join(data_dir, file_name), map_location='cpu', weights_only=False)
                data_list.append(data)
            except:
                continue
    print(f" 총 {len(data_list)}명 로드 완료. \n")
    return data_list

def evaluate_gat(data_list, group_name="AAL 그룹", epochs=50):
    if len(data_list) == 0: return 0, 0
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    labels = [int(data.y.item()) for data in data_list]
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    acc_scores, f1_scores = [], []
    print(f" {group_name} - [GAT] 5-Fold CV 시작 (Epochs: {epochs})")

    for fold, (train_idx, test_idx) in enumerate(skf.split(data_list, labels)):
        train_data = [data_list[i] for i in train_idx]
        test_data = [data_list[i] for i in test_idx]
        
        train_loader = DataLoader(train_data, batch_size=16, shuffle=True)
        test_loader = DataLoader(test_data, batch_size=16, shuffle=False)

        input_dim = data_list[0].num_features 
        model = BrainGAT(in_dim=input_dim, hidden_dim=64, num_classes=2, heads=4).to(device)
            
        optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=5e-4)
        criterion = torch.nn.CrossEntropyLoss()
        
        model.train()
        for epoch in range(epochs):
            for batch in train_loader:
                batch = batch.to(device)
                optimizer.zero_grad()
                out = model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)
                loss = criterion(out, batch.y)
                loss.backward()
                optimizer.step()
                
        model.eval()
        preds, trues = [], []
        with torch.no_grad():
            for batch in test_loader:
                batch = batch.to(device)
                out = model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)
                preds.extend(out.argmax(dim=1).cpu().numpy())
                trues.extend(batch.y.cpu().numpy())
                
        acc = accuracy_score(trues, preds)
        f1 = f1_score(trues, preds, average='macro')
        acc_scores.append(acc)
        f1_scores.append(f1)
        
    mean_acc, mean_f1 = np.mean(acc_scores), np.mean(f1_scores)
    print(f" {group_name} [GAT] 최종 평균 -> Acc: {mean_acc:.4f} | F1: {mean_f1:.4f}\n")
    return mean_acc, mean_f1

if __name__ == "__main__":
    BASE_DIR = r'C:\Users\rty76\OneDrive\바탕 화면\ASD_project'

    DIR_AAL_CHILD = os.path.join(BASE_DIR, 'data_AAL_Adolescent_Top20')
    DIR_AAL_ADULT = os.path.join(BASE_DIR, 'data_AAL_Adult_Top20')

    aal_child_data = load_prepared_data(DIR_AAL_CHILD)
    aal_adult_data = load_prepared_data(DIR_AAL_ADULT)

    child_acc, child_f1 = evaluate_gat(aal_child_data, group_name="AAL 116 청소년", epochs=50)
    adult_acc, adult_f1 = evaluate_gat(aal_adult_data, group_name="AAL 116 성인", epochs=50)
    
    # 3. 결과 브리핑
    print("====================================================")
    print(" [AAL 116 기준 GAT 연령대별 실험 결과 요약 Table]")
    print(f"▶ 청소년 (18세 이하) | Acc: {child_acc:.4f} | F1: {child_f1:.4f}")
    print(f"▶ 성인   (19세 이상) | Acc: {adult_acc:.4f} | F1: {adult_f1:.4f}")
    print("====================================================")